# Update a Sigma connection writeback location

This notebook resolves a Sigma connection by exact name, retrieves its current state, converts those values into a PUT payload, and adds a Databricks writeback catalog and schema. Sigma API credentials and the one non-retrievable OAuth client secret are read from a Databricks secret scope.

Run with `dry_run=true` first and review the generated payload before changing it to `false`.

In [ ]:
# Databricks widgets
dbutils.widgets.text("connection_name", "", "Sigma connection name")
dbutils.widgets.text("catalog", "", "New writeback catalog")
dbutils.widgets.text("schema", "", "New writeback schema")
dbutils.widgets.text("secret_scope", "sigma", "Databricks secret scope")
dbutils.widgets.text("client_id_secret_key", "sigma_client_id", "Client ID secret key")
dbutils.widgets.text("client_secret_secret_key", "sigma_client_secret", "Client secret key")
dbutils.widgets.text("databricks_oauth_client_secret_secret_key", "databricks_oauth_client_secret", "Databricks OAuth client secret key")
dbutils.widgets.text("sigma_base_url", "https://aws-api.sigmacomputing.com", "Sigma API base URL")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "Dry run")

In [ ]:
CONNECTION_NAME = dbutils.widgets.get("connection_name").strip()
WRITEBACK_CATALOG = dbutils.widgets.get("catalog").strip()
WRITEBACK_SCHEMA = dbutils.widgets.get("schema").strip()
SECRET_SCOPE = dbutils.widgets.get("secret_scope").strip()
CLIENT_ID_SECRET_KEY = dbutils.widgets.get("client_id_secret_key").strip()
CLIENT_SECRET_SECRET_KEY = dbutils.widgets.get("client_secret_secret_key").strip()
DATABRICKS_OAUTH_CLIENT_SECRET_SECRET_KEY = dbutils.widgets.get("databricks_oauth_client_secret_secret_key").strip()
SIGMA_BASE_URL = dbutils.widgets.get("sigma_base_url").strip().rstrip("/")
DRY_RUN = dbutils.widgets.get("dry_run").lower() == "true"

if not CONNECTION_NAME or not WRITEBACK_CATALOG or not WRITEBACK_SCHEMA:
    raise ValueError("connection_name, catalog, and schema widgets are required.")

print(f"Connection: {CONNECTION_NAME!r}")
print(f"Writeback: {WRITEBACK_CATALOG}.{WRITEBACK_SCHEMA}")
print(f"Dry run: {DRY_RUN}")

## Generated connection payload

The notebook builds the update payload from the connection returned by Sigma. No connection template needs to be maintained. Sigma does not return the connection-level OAuth client secret, so that value is loaded from the configured secret scope when required.

In [ ]:
# The payload is generated after the current connection is retrieved.


In [ ]:
import json
from copy import deepcopy
from typing import Any

import requests


def get_access_token() -> str:
    client_id = dbutils.secrets.get(SECRET_SCOPE, CLIENT_ID_SECRET_KEY)
    client_secret = dbutils.secrets.get(SECRET_SCOPE, CLIENT_SECRET_SECRET_KEY)
    response = requests.post(
        f"{SIGMA_BASE_URL}/v2/auth/token",
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
        },
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    token = data.get("access_token") or data.get("accessToken")
    if not token:
        raise RuntimeError("Sigma authentication response did not include an access token.")
    return token


def resolve_connection_id(token: str, connection_name: str) -> str:
    matches = []
    page = None
    headers = {"Authorization": f"Bearer {token}"}
    while True:
        params: dict[str, Any] = {"limit": 1000, "search": connection_name}
        if page:
            params["page"] = page
        response = requests.get(
            f"{SIGMA_BASE_URL}/v2/connections", headers=headers, params=params, timeout=30
        )
        response.raise_for_status()
        data = response.json()
        matches.extend(
            connection
            for connection in data.get("entries", [])
            if connection.get("name") == connection_name
            and not connection.get("isArchived", False)
        )
        page = data.get("nextPage")
        if not page:
            break

    if not matches:
        raise ValueError(f"No active Sigma connection found with name {connection_name!r}.")
    if len(matches) > 1:
        raise ValueError(f"Multiple active connections are named {connection_name!r}.")
    connection_id = matches[0].get("connectionId") or matches[0].get("id")
    if not connection_id:
        raise RuntimeError("Matched connection did not include a connection ID.")
    return str(connection_id)


def get_connection(token: str, connection_id: str) -> dict[str, Any]:
    response = requests.get(
        f"{SIGMA_BASE_URL}/v2/connections/{connection_id}",
        headers={"Authorization": f"Bearer {token}"},
        timeout=30,
    )
    response.raise_for_status()
    current = response.json()
    if not isinstance(current, dict):
        raise RuntimeError("Sigma connection response was not a JSON object.")
    return current


def build_updated_payload(current: dict[str, Any]) -> dict[str, Any]:
    if str(current.get("type", "")).lower() != "databricks":
        raise ValueError("Retrieved connection is not a Databricks connection.")
    host = current.get("account") or current.get("host")
    endpoint = current.get("warehouse") or current.get("endpoint")
    if not host:
        raise ValueError("Retrieved Databricks connection is missing its host/account value.")
    details: dict[str, Any] = {
        "type": "databricks",
        "host": host,
        "useOauth": bool(current.get("useOauth")),
    }
    if endpoint is not None:
        details["endpoint"] = endpoint
    user_attributes = current.get("userAttributes")
    if user_attributes is not None:
        if not isinstance(user_attributes, dict):
            raise ValueError("Retrieved userAttributes must be an object.")
        if user_attributes:
            details["userAttributes"] = deepcopy(user_attributes)
    for field in (
        "materializationWarehouse",
        "exportsWarehouse",
        "inputTableAuditLogSchema",
        "roleSwitching",
    ):
        if current.get(field) is not None:
            details[field] = deepcopy(current[field])
    if details["useOauth"]:
        independent_oauth = bool(current.get("isIndependentOAuth"))
        details["useOrgOauth"] = not independent_oauth
        if independent_oauth:
            details["oauth"] = {
                "provider": "databricks",
                "clientId": current.get("oauthClientId"),
                "clientSecret": {
                    "type": "plain",
                    "value": dbutils.secrets.get(
                        SECRET_SCOPE, DATABRICKS_OAUTH_CLIENT_SECRET_SECRET_KEY
                    ),
                },
                "metadataUrl": current.get("oauthMetadataUrl"),
                "scopes": current.get("oauthScopes") or [],
            }
    fetched = current.get("writebackSchemas", [])
    if not isinstance(fetched, list):
        raise ValueError("Retrieved writebackSchemas must be an array.")
    schemas = []
    for entry in fetched:
        catalog = entry.get("writeCatalog") or entry.get("database") or entry.get("catalog")
        schema = entry.get("writeSchema") or entry.get("schema")
        if not catalog or not schema:
            raise ValueError(f"Retrieved writeback entry is incomplete: {entry!r}")
        location = {"writeCatalog": catalog, "writeSchema": schema}
        if entry.get("description") is not None:
            location["description"] = entry["description"]
        schemas.append(location)
    location = {"writeCatalog": WRITEBACK_CATALOG, "writeSchema": WRITEBACK_SCHEMA}
    if any(s["writeCatalog"] == WRITEBACK_CATALOG and s["writeSchema"] == WRITEBACK_SCHEMA for s in schemas):
        raise ValueError(f"Writeback location {WRITEBACK_CATALOG}.{WRITEBACK_SCHEMA} already exists.")
    schemas.append(location)
    details["writebackSchemas"] = schemas
    payload: dict[str, Any] = {"name": current["name"], "details": details}
    if current.get("description") is not None:
        payload["description"] = deepcopy(current["description"])
    if current.get("poolSizes") is not None:
        payload["poolSizes"] = deepcopy(current["poolSizes"])
    timeout = current.get("timeout")
    if isinstance(timeout, dict) and timeout.get("default") is not None:
        payload["timeoutSecs"] = timeout["default"]
    if current.get("friendlyName") is not None:
        payload["useFriendlyNames"] = current["friendlyName"]
    return payload

In [ ]:
access_token = get_access_token()
connection_id = resolve_connection_id(access_token, CONNECTION_NAME)
current_connection = get_connection(access_token, connection_id)
updated_payload = build_updated_payload(current_connection)
print(json.dumps(updated_payload, indent=2))

if DRY_RUN:
    print("[DRY RUN] Retrieved the connection; no Sigma API update was submitted.")
else:
    response = requests.put(
        f"{SIGMA_BASE_URL}/v2/connections/{connection_id}",
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        },
        json=updated_payload,
        timeout=30,
    )
    response.raise_for_status()
    result = response.json()
    print(f"[OK] Updated {CONNECTION_NAME!r} ({connection_id}).")
    print(json.dumps(result.get("writebackSchemas") or result.get("writebacks") or [], indent=2))